# N167 · 死锁、饥饿与哲学家问题

**目标：** 区分安全性、无死锁和有条件的无饥饿。

**先修：** N164, N165, N166, N013。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 资源顺序；循环等待；FIFO准入；公平调度假设；有限持有时间；活性。

## 从问题到算法

“没有死锁”不等于“没有饥饿”。本章采用 FIFO 请求队列，在一把条件锁内同时预留两把叉子：不允许线程拿一把叉子再等另一把，因此消除循环等待；FIFO 则阻止后来的请求无限插队。

队列单位必须是**请求票号**，不是哲学家编号。同一位哲学家可以有多个并发调用，每个调用必须拥有独立的等待、授予和释放记录。

严格 FIFO 会产生队头阻塞：即使后面某个请求能立即获得两把空闲叉子，也可能等待队头。这是可解释公平性与吞吐量的取舍。下面的检查器重放真实资源轨迹，验证独占、持有两叉再吃、准确释放以及票号顺序。

## 最小实现

**本章接口：** `DiningPhilosophersFIFO.wantsToEat(...)`、`validate_fork_trace(events)`

In [1]:
THREAD_SOURCE = r'''import threading
from collections import deque

def run_threads(functions):
    """Caller must run potentially blocking cases inside the isolated process."""
    errors = []
    error_lock = threading.Lock()
    def guarded(fn):
        try:
            fn()
        except BaseException as error:
            with error_lock:
                errors.append(error)
    threads = [threading.Thread(target=guarded, args=(fn,)) for fn in functions]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    if errors:
        raise errors[0]
    assert not any(thread.is_alive() for thread in threads)

class DiningPhilosophersFIFO:
    def __init__(self):
        self.condition = threading.Condition()
        self.waiting = deque()
        self.next_ticket = 0
        self.owners = [None]*5
        self.events = []
    def _event(self,action,ticket,philosopher,payload=None):
        self.events.append((action,ticket,philosopher,payload))
    def wantsToEat(self,philosopher,pickLeftFork,pickRightFork,eat,putLeftFork,putRightFork):
        if philosopher not in range(5):
            raise ValueError('Philosopher must be 0..4')
        left,right = philosopher,(philosopher+1)%5
        with self.condition:
            ticket = self.next_ticket
            self.next_ticket += 1
            self.waiting.append((ticket,philosopher))
            self._event('request',ticket,philosopher)
            self.condition.notify_all()
            while self.waiting[0][0] != ticket or self.owners[left] is not None or self.owners[right] is not None:
                self.condition.wait()
            self.waiting.popleft()
            self.owners[left] = self.owners[right] = ticket
            self._event('grant',ticket,philosopher,(left,right))
            self.condition.notify_all()
        try:
            pickLeftFork()
            with self.condition:self._event('pick',ticket,philosopher,left)
            pickRightFork()
            with self.condition:self._event('pick',ticket,philosopher,right)
            eat()
            with self.condition:self._event('eat',ticket,philosopher)
            putLeftFork()
            with self.condition:self._event('put',ticket,philosopher,left)
            putRightFork()
            with self.condition:self._event('put',ticket,philosopher,right)
        finally:
            with self.condition:
                self.owners[left] = self.owners[right] = None
                self._event('release',ticket,philosopher,(left,right))
                self.condition.notify_all()
        return ticket

def validate_fork_trace(events):
    waiting = deque()
    records = {}
    owners = [None]*5
    for action,ticket,p,payload in events:
        forks = {p,(p+1)%5}
        if action=='request':
            assert ticket not in records
            records[ticket] = {'p':p,'picked':set(),'put':set(),'eaten':0,'released':False,'granted':False}
            waiting.append(ticket)
            continue
        assert ticket in records and records[ticket]['p']==p
        record = records[ticket]
        assert not record['released']
        if action=='grant':
            assert waiting and waiting.popleft()==ticket
            assert not record['granted'] and set(payload)==forks
            assert all(owners[f] is None for f in forks)
            for f in forks:owners[f]=ticket
            record['granted']=True
        elif action=='pick':
            assert record['granted'] and payload in forks and owners[payload]==ticket
            assert payload not in record['picked']
            record['picked'].add(payload)
        elif action=='eat':
            assert record['picked']==forks and not record['put'] and record['eaten']==0
            assert all(owners[f]==ticket for f in forks)
            record['eaten']=1
        elif action=='put':
            assert record['eaten']==1 and payload in forks and payload not in record['put']
            assert owners[payload]==ticket
            record['put'].add(payload)
        elif action=='release':
            assert record['put']==forks and record['eaten']==1 and set(payload)==forks
            assert all(owners[f]==ticket for f in forks)
            for f in forks:owners[f]=None
            record['released']=True
        else:
            raise AssertionError('Unknown trace event')
    assert not waiting and all(x is None for x in owners)
    assert all(r['released'] for r in records.values())
    return True
'''

import json
import subprocess
import sys

# This defines the visible APIs only; it does not launch a thread.
exec(THREAD_SOURCE, globals())

def run_isolated(extra):
    """A timeout is a failure; subprocess.run kills and reaps the child."""
    result = subprocess.run([sys.executable, '-c', THREAD_SOURCE + '\n' + extra],
        text=True, capture_output=True, check=True, timeout=20)
    return result.stdout

## 正确性、前提与复杂度

安全性：grant 在同一锁内同时检查并设置两个 owner，任何叉子不会属于两个票号。无死锁：已获准请求不再等待叉子；它有限完成后释放，队头最终可进。无饥饿需要有限回调、锁与线程弱公平调度，以及每个已排队请求之前仅有有限个请求；按票号归纳，前序请求终会退出。有限压测只检查此次运行，不代替这些活性前提。回调异常会向外传播；轨迹不完整不会被检查器视为成功。

**代价：** 叉子状态 O(1)，FIFO 队列 O(q)。本章为核验保存 O(r) 条事件；条件通知可能唤醒多个等待者，不声称每个请求的总调度开销为 O(1)。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

rows = json.loads(run_isolated("""import json
obj=DiningPhilosophersFIFO()
callbacks=[lambda:None]*5
run_threads([lambda p=p:obj.wantsToEat(p,*callbacks) for p in [0,2,4,1,3]])
assert validate_fork_trace(obj.events)
print(json.dumps(obj.events))
"""))
show_table(['event','ticket','philosopher','forks'],rows)

event,ticket,philosopher,forks
request,0,0,None
grant,0,0,"[0, 1]"
pick,0,0,0
pick,0,0,1
eat,0,0,None
put,0,0,0
put,0,0,1
release,0,0,"[0, 1]"
request,1,2,None
grant,1,2,"[2, 3]"


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
print(run_isolated("""from collections import Counter
for _ in range(3):
    obj=DiningPhilosophersFIFO()
    first_started=threading.Event();release_first=threading.Event();go=threading.Event()
    seen=Counter();seen_lock=threading.Lock();tickets=[]
    requests=[0,0,0,1,2,3,4,0,2,4]
    def request(i,p):
        if i:go.wait()
        def eating():
            if i==0:
                first_started.set();release_first.wait()
            with seen_lock:seen[i]+=1
        ticket=obj.wantsToEat(p,lambda:None,lambda:None,eating,lambda:None,lambda:None)
        with seen_lock:tickets.append(ticket)
    def coordinate():
        first_started.wait();go.set()
        with obj.condition:
            while obj.next_ticket != len(requests):obj.condition.wait()
        release_first.set()
    run_threads([lambda i=i,p=p:request(i,p) for i,p in enumerate(requests)]+[coordinate])
    assert validate_fork_trace(obj.events)
    assert seen==Counter({i:1 for i in range(len(requests))})
    assert len(set(tickets))==len(requests)
    bad=list(obj.events)
    bad.insert(1,('eat',0,0,None))
    try:validate_fork_trace(bad)
    except AssertionError:pass
    else:raise AssertionError('Eating without forks was accepted')
assert threading.active_count()==1
print('FIFO tickets, concurrent repeated philosophers, ownership and cleanup passed.')
"""))
print('N167: 所有本章断言通过')

FIFO tickets, concurrent repeated philosophers, ownership and cleanup passed.

N167: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 比较统一加锁顺序和FIFO准入：前者无死锁不自动证明无饥饿。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 为活性明确调度假设。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 1226. The Dining Philosophers（canonical）：[官方入口](https://leetcode.com/problems/the-dining-philosophers/)
- 1195. Fizz Buzz Multithreaded（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。